# Lesson 9: Autoencoders and latent space — squeeze, then create 🗜️
**Learning goals**
1. Understand an **autoencoder**: compress (encode) → tiny code → rebuild (decode)
2. Draw and explore a **latent space** — a "map" where similar things sit close together
3. Train a **VAE** and use it to generate brand-new digits
4. See a **denoising autoencoder** — a stepping stone to diffusion models (next lesson)

**⏱️ Timing:** autoencoder 15' | latent map 15' | VAE 20' | denoising 10'

> ⚙️ GPU recommended: **Runtime → Change runtime type → T4 GPU**. CPU also works (about 1–2 minutes per training run).

## 🗣️ An analogy: describe a friend in two words
Imagine you must describe a friend to a sketch artist using **only two numbers**, e.g. *height* and *hair length*. The artist draws from those two numbers alone.
- The **encoder** = you, squeezing a whole person into 2 numbers
- The **latent code** = the 2 numbers (the "bottleneck")
- The **decoder** = the artist, rebuilding a picture from the code

To do this well, you're forced to pick the **most important** features. An autoencoder learns that by itself: it's trained only to make the **output match the input**.

**Latent space** = the space of all possible codes. "Latent" means **hidden** — features like *height* aren't written in the pixels, the network discovers them.

In [ ]:
import torch, torch.nn as nn, numpy as np, time
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
mnist = datasets.MNIST(".", train=True, download=True, transform=transforms.ToTensor())
loader = torch.utils.data.DataLoader(mnist, batch_size=256, shuffle=True)
test_imgs = torch.stack([mnist[i][0] for i in range(5000)]).view(-1, 784).to(device)
test_labels = torch.tensor([mnist[i][1] for i in range(5000)])
print("device:", device, "| each image = 28 × 28 =", 28 * 28, "numbers")

## Part 1: Train an autoencoder: 784 numbers → 2 → 784
That's squeezing each picture down to **0.26%** of its size!

In [ ]:
LATENT = 2

encoder = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, LATENT)).to(device)
decoder = nn.Sequential(nn.Linear(LATENT, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid()).to(device)
opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)

start = time.time()
for epoch in range(5):
    for x, _ in loader:
        x = x.view(-1, 784).to(device)
        rebuilt = decoder(encoder(x))
        loss = nn.functional.mse_loss(rebuilt, x)          # how different is the copy from the original?
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch + 1}: loss {loss.item():.4f}  ({time.time() - start:.0f}s)")

In [ ]:
def compare(model_fn, title, n=10):
    with torch.no_grad():
        out = model_fn(test_imgs[:n]).cpu()
    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 2.6))
    for i in range(n):
        axes[0, i].imshow(test_imgs[i].cpu().view(28, 28), cmap="gray"); axes[0, i].axis("off")
        axes[1, i].imshow(out[i].view(28, 28), cmap="gray"); axes[1, i].axis("off")
    axes[0, 0].set_title("original", loc="left"); axes[1, 0].set_title("rebuilt from the code", loc="left")
    plt.suptitle(title); plt.show()

compare(lambda x: decoder(encoder(x)), "autoencoder with a 2-number bottleneck")
with torch.no_grad():
    print("The secret code for the first picture is just:", encoder(test_imgs[:1]).cpu().numpy().round(2))

### 🧩 Exercise 1
1. The rebuilt digits are blurry. Why? (Two numbers can't store every detail.)
2. Change `LATENT = 2` to `LATENT = 16`, retrain, and compare. Clearer? Why?
3. Where have you seen compression before? (JPG photos, MP3 music, ZIP files…) How is an autoencoder different? (It **learns** what to keep for one kind of data.)

⚠️ Set `LATENT` back to **2** before continuing, so we can draw the map.

## Part 2: Draw the latent space map 🗺️
Every picture becomes a point (x, y). Let's colour each point by which digit it really is. **Nobody told the network the labels!**

In [ ]:
def draw_map(enc_fn, title):
    with torch.no_grad():
        z = enc_fn(test_imgs).cpu().numpy()
    plt.figure(figsize=(8, 7))
    sc = plt.scatter(z[:, 0], z[:, 1], c=test_labels, cmap="tab10", s=4)
    plt.colorbar(sc, ticks=range(10)); plt.title(title); plt.xlabel("latent x"); plt.ylabel("latent y")
    plt.show()
    return z

z = draw_map(encoder, "autoencoder latent space (colour = true digit)")

### 🧩 Exercise 2: Read the map
1. Which digits form their own **islands**? Which ones **overlap** (look similar to the network)?
2. Do the overlaps make sense? (4 and 9? 3, 5 and 8?)

Like a city map where all the bakeries are in one street: **similar things end up close together**. This is what "latent space" means in every generative model — including the word and image spaces inside ChatGPT and image generators.

## Part 3: Walk the map — decode any point 🚶
Pick a point, and the decoder draws whatever lives there.

In [ ]:
def explore(dec, z, n=12):
    xs = np.linspace(np.percentile(z[:, 0], 2), np.percentile(z[:, 0], 98), n)
    ys = np.linspace(np.percentile(z[:, 1], 98), np.percentile(z[:, 1], 2), n)
    grid = torch.tensor([[x, y] for y in ys for x in xs], dtype=torch.float32, device=device)
    with torch.no_grad():
        imgs = dec(grid).cpu().view(n, n, 28, 28)
    big = imgs.permute(0, 2, 1, 3).reshape(n * 28, n * 28)
    plt.figure(figsize=(8, 8)); plt.imshow(big, cmap="gray"); plt.axis("off"); plt.title("decoding a grid of points"); plt.show()

explore(decoder, z)

In [ ]:
# 👉 Your turn: choose any (x, y) from the map above and see what the decoder draws
point = torch.tensor([[0.0, 0.0]], device=device)    # change these two numbers!
with torch.no_grad():
    plt.imshow(decoder(point).cpu().view(28, 28), cmap="gray"); plt.axis("off"); plt.show()

### 🧩 Exercise 3
1. Find a point that makes a clear **7**, then one that makes a **"weird" mix** of two digits.
2. Pick a point **far outside** all the islands. What does the decoder draw? This is the autoencoder's big weakness — the map has **holes** where nothing sensible lives.

## Part 4: The VAE — a map with no holes 🌐
A **Variational Autoencoder (VAE)** changes one thing: instead of one exact point, the encoder outputs a **small cloud** (a centre and a spread), and we pick a random point inside it. A second rule pushes all the clouds to gather around (0, 0) in a neat round shape.

> Analogy: instead of pinning each friend to one exact spot on the map, you say "somewhere around here". Neighbouring areas overlap and fill in, so **every** point decodes into something sensible — and you can generate new things by picking random points!

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU())
        self.mu = nn.Linear(64, LATENT)          # centre of the cloud
        self.logvar = nn.Linear(64, LATENT)      # spread of the cloud
        self.dec = nn.Sequential(nn.Linear(LATENT, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(),
                                 nn.Linear(256, 784), nn.Sigmoid())
    def encode(self, x):
        h = self.body(x)
        return self.mu(h), self.logvar(h)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)   # pick a random point in the cloud
        return self.dec(z), mu, logvar

vae = VAE().to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
start = time.time()
for epoch in range(8):
    for x, _ in loader:
        x = x.view(-1, 784).to(device)
        out, mu, logvar = vae(x)
        rebuild = nn.functional.binary_cross_entropy(out, x, reduction="sum") / len(x)
        tidy = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(x)   # keeps clouds round and near (0,0)
        loss = rebuild + tidy
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch + 1}: rebuild {rebuild.item():.1f}  tidiness {tidy.item():.2f}  ({time.time() - start:.0f}s)")

In [ ]:
z_vae = draw_map(lambda x: vae.encode(x)[0], "VAE latent space: neater and centred on (0, 0)")
explore(vae.dec, z_vae)

### ✨ Generate brand-new digits
Pick **random** points near (0, 0) and decode them. These digits were never in the training data!

In [ ]:
with torch.no_grad():
    new = vae.dec(torch.randn(24, LATENT, device=device)).cpu()
fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for ax, im in zip(axes.flat, new):
    ax.imshow(im.view(28, 28), cmap="gray"); ax.axis("off")
plt.suptitle("brand-new digits from random points"); plt.show()

### 🧩 Exercise 4: Morphing
Encode two real digits, then walk in a straight line between their codes. (You'll do the same with a GAN's random numbers in Lesson 11 — that's also a walk through latent space!)

In [ ]:
a, b = 0, 1     # 👉 indexes of two test pictures (try different ones)
with torch.no_grad():
    za, zb = vae.encode(test_imgs[a:a+1])[0], vae.encode(test_imgs[b:b+1])[0]
    steps = [vae.dec((1 - t) * za + t * zb).cpu() for t in np.linspace(0, 1, 10)]
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, im in zip(axes, steps):
    ax.imshow(im.view(28, 28), cmap="gray"); ax.axis("off")
plt.suptitle(f"morphing a {test_labels[a].item()} into a {test_labels[b].item()}"); plt.show()

## Part 5: Denoising autoencoder — a sneak peek at diffusion 🧽
Train the autoencoder with **noisy** pictures as input, but the **clean** picture as the target. It learns to remove noise — and **removing noise is exactly what diffusion models do** (next lesson!).

In [ ]:
den_enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU()).to(device)
den_dec = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid()).to(device)
opt = torch.optim.Adam(list(den_enc.parameters()) + list(den_dec.parameters()), lr=1e-3)
NOISE = 0.5      # 👉 try 0.3 or 0.8

def add_noise(x): return (x + NOISE * torch.randn_like(x)).clamp(0, 1)

for epoch in range(3):
    for x, _ in loader:
        x = x.view(-1, 784).to(device)
        loss = nn.functional.mse_loss(den_dec(den_enc(add_noise(x))), x)   # noisy in, clean target
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch + 1}: loss {loss.item():.4f}")

noisy = add_noise(test_imgs[:10])
with torch.no_grad():
    cleaned = den_dec(den_enc(noisy)).cpu()
fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for i in range(10):
    axes[0, i].imshow(noisy[i].cpu().view(28, 28), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(cleaned[i].view(28, 28), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_title("noisy input", loc="left"); axes[1, 0].set_title("cleaned by the network", loc="left"); plt.show()

### 🚀 Challenge
- Make `NOISE` really big (1.5). Can it still recover the digits? A diffusion model solves this by removing noise in **many small steps** instead of one big one.
- Try `LATENT = 3` for the VAE and plot two of the three dimensions at a time.

## 📌 Summary
- **Autoencoder** = encoder (squeeze) → latent code → decoder (rebuild), trained to copy its input
- **Latent space** = a hidden "map" where similar things sit close together; you can walk it and morph between things
- **VAE** = encodes to clouds, giving a smooth map with no holes → you can **generate** new things from random points
- **Denoising autoencoder** learns to remove noise → the key idea behind **diffusion models** (next lesson)
- Big image generators like Stable Diffusion actually run diffusion **inside an autoencoder's latent space** — that's why they're called *latent* diffusion models!